In [23]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder
from sklearn.preprocessing import OneHotEncoder

import os 
os.chdir(r'E:\Backup Files Abhinav_Satyam\Practical Machine Learning\Sanjay Sir Git Repo\Datasets')
import warnings
warnings.filterwarnings("ignore")

In [24]:
housing = pd.read_csv('Housing.csv')
housing

,price,lotsize,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea
0,42000.0,5850,3,1,2,yes,no,yes,no,no,1,no
1,38500.0,4000,2,1,1,yes,no,no,no,no,0,no
2,49500.0,3060,3,1,1,yes,no,no,no,no,0,no
3,60500.0,6650,3,1,2,yes,yes,no,no,no,0,no
4,61000.0,6360,2,1,1,yes,no,no,no,no,0,no
...,...,...,...,...,...,...,...,...,...,...,...,...
541,91500.0,4800,3,2,4,yes,yes,no,no,yes,0,no
542,94000.0,6000,3,2,4,yes,no,no,no,yes,0,no
543,103000.0,6000,3,2,4,yes,yes,no,no,yes,1,no
544,105000.0,6000,3,2,2,yes,yes,no,no,yes,1,no


# transform the dataset in to proper format containing boolean values

In [25]:
intervals = [(0,50000),(50000,100000),(100000, 150000) , (150000, 200000)]
bins = pd.IntervalIndex.from_tuples(intervals)
housing['price_slabs'] = pd.cut(housing['price'],bins)

intervals = [(0,5000),(5000,10000),(10000, 15000) , (15000, 20000)]
bins = pd.IntervalIndex.from_tuples(intervals)
housing['area_slabs'] = pd.cut(housing['lotsize'],bins)


housing.drop(['price','lotsize'], axis=1, inplace=True)

In [28]:
ohc = OneHotEncoder(sparse_output=False, dtype=np.bool_).set_output(transform='pandas')
housing_trans = ohc.fit_transform(housing)
housing_trans

,bedrooms_1,bedrooms_2,bedrooms_3,bedrooms_4,bedrooms_5,bedrooms_6,bathrms_1,bathrms_2,bathrms_3,bathrms_4,...,prefarea_no,prefarea_yes,"price_slabs_(0, 50000]","price_slabs_(50000, 100000]","price_slabs_(100000, 150000]","price_slabs_(150000, 200000]","area_slabs_(0, 5000]","area_slabs_(5000, 10000]","area_slabs_(10000, 15000]","area_slabs_(15000, 20000]"
0,False,False,True,False,False,False,True,False,False,False,...,True,False,True,False,False,False,False,True,False,False
1,False,True,False,False,False,False,True,False,False,False,...,True,False,True,False,False,False,True,False,False,False
2,False,False,True,False,False,False,True,False,False,False,...,True,False,True,False,False,False,True,False,False,False
3,False,False,True,False,False,False,True,False,False,False,...,True,False,False,True,False,False,False,True,False,False
4,False,True,False,False,False,False,True,False,False,False,...,True,False,False,True,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,False,False,True,False,False,False,False,True,False,False,...,True,False,False,True,False,False,True,False,False,False
542,False,False,True,False,False,False,False,True,False,False,...,True,False,False,True,False,False,False,True,False,False
543,False,False,True,False,False,False,False,True,False,False,...,True,False,False,False,True,False,False,True,False,False
544,False,False,True,False,False,False,False,True,False,False,...,True,False,False,False,True,False,False,True,False,False


 # Applying apriori on it

In [43]:
itemsets = apriori(housing_trans, min_support=0.6,use_colnames=True)

rules = association_rules(itemsets , metric='confidence' , min_threshold=0.7)
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift',]]
rules.sort_values(by=['confidence','lift','support'], ascending=False)

relv_rules= rules[rules['lift']>1]
relv_rules.sort_values(by=['confidence','lift','support'], ascending=False)


,antecedents,consequents,support,confidence,lift
24,"(driveway_yes, bathrms_1)",(gashw_no),0.602564,0.964809,1.011105
5,(bathrms_1),(gashw_no),0.708791,0.962687,1.008881
29,"(recroom_no, bathrms_1)",(gashw_no),0.602564,0.959184,1.005210
9,(driveway_yes),(gashw_no),0.820513,0.955224,1.001060
34,"(driveway_yes, recroom_no)",(gashw_no),0.663004,0.955145,1.000977
18,(fullbase_no),(gashw_no),0.620879,0.954930,1.000752
12,(fullbase_no),(recroom_no),0.602564,0.926761,1.126974
8,(gashw_no),(driveway_yes),0.820513,0.859885,1.001060
37,"(prefarea_no, gashw_no)",(recroom_no),0.622711,0.858586,1.044071
16,(prefarea_no),(recroom_no),0.655678,0.856459,1.041485


In [32]:
from ipywidgets import interact
import ipywidgets as widgets

In [35]:
def gen_rules(min_support , minconfidence):
    itemsets = apriori(housing_trans, min_support=min_support, use_colnames=True)
    rules = association_rules(itemsets , metric='confidence' , min_threshold=minconfidence)
    rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift',]]
    relv_rules= rules[rules['lift']>1]
    return relv_rules.sort_values(by=['confidence','lift','support'], ascending=False)


In [36]:
interact(
    gen_rules , 
    min_support = widgets.FloatSlider(
        value = 0.2 , min = 0.01, max = 0.4 , step = 0.01 , description = 'Min Support:'
    ),
    minconfidence = widgets.FloatSlider(
        value = 0.2 , min = 0.01, max = 0.5 , step = 0.01, description = 'Min Confidence:'
    )
)

interactive(children=(FloatSlider(value=0.2, description='Min Support:', max=0.4, min=0.01, step=0.01), FloatS…

<function __main__.gen_rules(min_support, minconfidence)>